#### 서울시 구별 주유소 유가 정보 획득

Selenium으로 화면을 조작하고 BeautifulSoup으로 렌더링된 HTML의 자치구 목록을 수집합니다. 위에서부터 순서대로 실행하세요. Chrome이 필요하며, 최초 실행 시 Selenium Manager가 드라이버를 준비합니다.

### 1. 라이브러리 설치
최초 한 번 실행합니다.

In [ ]:
%pip install selenium beautifulsoup4


### 2. 라이브러리 및 다운로드 경로 설정

In [ ]:
import time
from pathlib import Path
from datetime import datetime
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Note 폴더 또는 프로젝트 루트에서 실행할 수 있습니다.
ROOT = Path.cwd().parent if Path.cwd().name == 'Note' else Path.cwd()
DOWNLOAD_DIR = ROOT / 'Data' / 'opinet' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
URL = 'https://www.opinet.co.kr/searRgSelect.do'
PAGE_SLEEP = 3
print(DOWNLOAD_DIR)


### 3. Chrome 실행 및 사이트 접속
지역 선택 화면으로 연결되지 않으면 브라우저에서 ‘싼 주유소 찾기 → 지역별’을 연 후 다음 셀을 실행합니다.

In [ ]:
options = webdriver.ChromeOptions()
options.add_experimental_option('prefs', {
    'download.default_directory': str(DOWNLOAD_DIR.resolve()),
    'download.prompt_for_download': False,
    'download.directory_upgrade': True,
})
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 30)
driver.get(URL)
time.sleep(PAGE_SLEEP)
# 최초 접속 시 메인 화면으로 이동하는 경우 지역별 화면에 다시 접속합니다.
if not driver.find_elements(By.ID, 'SIDO_NM0'):
    driver.get(URL)
    time.sleep(PAGE_SLEEP)


### 4. 서울 선택 및 BeautifulSoup으로 25개 자치구 수집

In [ ]:
sido = wait.until(EC.presence_of_element_located((By.ID, 'SIDO_NM0')))
seoul_option = next(option for option in Select(sido).options if '서울' in option.text)
Select(sido).select_by_value(seoul_option.get_attribute('value'))
time.sleep(PAGE_SLEEP)
wait.until(lambda d: len(Select(d.find_element(By.ID, 'SIGUNGU_NM0')).options) >= 26)
soup = BeautifulSoup(driver.page_source, 'html.parser')
gu_list = [
    (option.get('value'), option.get_text(strip=True))
    for option in soup.select('#SIGUNGU_NM0 option')
    if option.get('value') and option.get_text(strip=True).endswith('구')
]
assert len(gu_list) == 25, f'자치구 수를 확인하세요: {len(gu_list)}개'
print([name for value, name in gu_list])


### 5. 다운로드 완료 확인 함수
다운로드 중인 임시 파일이 사라지고 엑셀 파일의 크기가 안정될 때까지 기다립니다.

In [ ]:
def wait_for_excel(before, timeout=90):
    deadline = time.monotonic() + timeout
    previous = None
    while time.monotonic() < deadline:
        files = set(DOWNLOAD_DIR.iterdir())
        candidates = [p for p in files - before if p.suffix.lower() in ('.xls', '.xlsx')]
        downloading = any(p.suffix.lower() in ('.crdownload', '.tmp') for p in files)
        if len(candidates) == 1 and not downloading:
            path = candidates[0]
            state = (path, path.stat().st_size)
            if state == previous and state[1] > 0:
                return path
            previous = state
        time.sleep(1)
    raise TimeoutError('엑셀 다운로드가 완료되지 않았습니다. 브라우저를 확인하세요.')


### 6. 자치구별 조회 및 화면 하단 엑셀 버튼 클릭
각 구를 선택한 뒤 조회 버튼을 누르고 화면 변경을 기다립니다. 오류가 발생하면 중단하며, 완료된 파일은 유지됩니다.

In [8]:
downloaded_files = []
for gu_value, gu_name in gu_list:
    Select(driver.find_element(By.ID, 'SIGUNGU_NM0')).select_by_value(gu_value)
    time.sleep(PAGE_SLEEP)
    wait.until(EC.element_to_be_clickable((By.ID, 'searRgSelect'))).click()
    time.sleep(PAGE_SLEEP)
    before = set(DOWNLOAD_DIR.iterdir())
    # 엑셀은 고정 ID 대신 이미지의 alt/src와 링크의 다운로드 동작으로 찾습니다.
    excel_xpath = (
        "//img[contains(@alt, '엑셀') or "
        "contains(translate(@src, 'EXCEL', 'excel'), 'excel')]"
        " | //a[contains(@href, 'Excel') or contains(@href, 'excel') "
        "or contains(@onclick, 'Excel') or contains(@onclick, 'excel')]"
        " | //button[contains(., '엑셀')] | //a[contains(., '엑셀')]"
    )

    def find_excel_button(browser):
        for element in browser.find_elements(By.XPATH, excel_xpath):
            if element.is_displayed() and element.is_enabled():
                # 이미지 자체보다 연결된 링크/버튼을 클릭합니다.
                if element.tag_name.lower() == 'img':
                    parents = element.find_elements(By.XPATH, './ancestor::*[self::a or self::button][1]')
                    if parents:
                        return parents[0]
                return element
        return False

    excel_button = wait.until(find_excel_button, '화면에서 엑셀 다운로드 버튼을 찾지 못했습니다.')
    driver.execute_script('arguments[0].scrollIntoView({block: "center"});', excel_button)
    excel_button.click()
    source = wait_for_excel(before)
    destination = DOWNLOAD_DIR / f'서울_{gu_name}{source.suffix}'
    source.rename(destination)
    downloaded_files.append(destination)
    print(f'{len(downloaded_files):02d}/25 {gu_name}: {destination.name}')
    time.sleep(1)


01/25 강남구: 서울_강남구.xls
02/25 강동구: 서울_강동구.xls
03/25 강북구: 서울_강북구.xls
04/25 강서구: 서울_강서구.xls
05/25 관악구: 서울_관악구.xls
06/25 광진구: 서울_광진구.xls
07/25 구로구: 서울_구로구.xls
08/25 금천구: 서울_금천구.xls
09/25 노원구: 서울_노원구.xls
10/25 도봉구: 서울_도봉구.xls
11/25 동대문구: 서울_동대문구.xls
12/25 동작구: 서울_동작구.xls
13/25 마포구: 서울_마포구.xls
14/25 서대문구: 서울_서대문구.xls
15/25 서초구: 서울_서초구.xls
16/25 성동구: 서울_성동구.xls
17/25 성북구: 서울_성북구.xls
18/25 송파구: 서울_송파구.xls
19/25 양천구: 서울_양천구.xls
20/25 영등포구: 서울_영등포구.xls
21/25 용산구: 서울_용산구.xls
22/25 은평구: 서울_은평구.xls
23/25 종로구: 서울_종로구.xls
24/25 중구: 서울_중구.xls
25/25 중랑구: 서울_중랑구.xls


### 7. 수집 결과 확인

In [9]:
assert len(downloaded_files) == 25, '25개 구의 다운로드 결과를 확인하세요.'
print(f'총 {len(downloaded_files)}개 엑셀 파일 저장 완료')
print(DOWNLOAD_DIR)


총 25개 엑셀 파일 저장 완료
c:\My\Python\Explorer\DataAnalysis\Data\opinet\20261112_185430_774766


### 8. 브라우저 종료
다운로드가 끝난 후 실행합니다.

In [10]:
driver.quit()
